# Étape 3 : contrôles de format, de complétude et de doublons

Ce notebook applique aux données chargées les règles du dictionnaire enregistré (`data/config/dictionnaire.yaml`), tel qu'il a été généré puis revu par le métier dans le notebook `03_dictionnaire`, ainsi que les règles complémentaires et les déclarations de doublons revues par le métier dans le formulaire des règles de contrôle, puis enregistrées dans `config/controles.yaml`.

Pour chaque colonne, c'est la règle en vigueur qui est contrôlée : champ obligatoire, nature, liste de valeurs, format. Un écart est une anomalie, ou reste à arbitrer quand le métier n'a pas encore décidé de la valeur ou de la forme.

Le notebook cherche ensuite les doublons : lignes identiques, valeurs répétées dans une colonne déclarée unique, lignes identiques hors de leur clé. Rien n'est supprimé, chaque doublon est une anomalie ou reste à arbitrer.

Avant les contrôles, trois cellules font circuler ce formulaire (phase 5 du mode opératoire). `# EXPORT FORMULAIRE` l'écrit dans `data/formulaires/` s'il n'y est pas déjà, et le métier le remplit dans Excel. `# IMPORT FORMULAIRE` puis `# ECRITURE FORMULAIRE` réécrivent alors `config/controles.yaml`, après une copie datée de l'ancien fichier, et archivent le formulaire appliqué dans `data/formulaires/appliques/`. Un formulaire pas encore rempli ne change rien, si bien que le notebook peut toujours être exécuté en entier.

Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `03_dictionnaire`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 03_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# EXPORT FORMULAIRE

from openpyxl import load_workbook

from frigeo.confidentialite import charger_sensibilite
from frigeo.controles import lire_fichier_controles
from frigeo.formulaire_controles import (
    FEUILLE_MODE_EMPLOI,
    chemin_formulaire,
    empreinte_controles,
    exporter_formulaire,
)

# Le formulaire part des règles en vigueur dans config/controles.yaml et des données
# chargées. Un formulaire déjà exporté n'est pas remplacé, il attend son import.
chemin_form = chemin_formulaire(PERIODE_FIN)
if chemin_form.exists():
    print("Formulaire déjà exporté, en attente d'import,", chemin_form)
else:
    exporter_formulaire(
        tables, dictionnaire, lire_fichier_controles(), charger_sensibilite(), PERIODE_FIN,
        empreinte=empreinte_controles(),
    )
    print("Formulaire exporté,", chemin_form)

# Ce qui reste à traiter, tel que le mode d'emploi du formulaire l'affiche.
classeur = load_workbook(chemin_form, read_only=True)
try:
    for (texte,) in classeur[FEUILLE_MODE_EMPLOI].iter_rows(max_col=1, values_only=True):
        if texte and texte.startswith("Feuille") and "observé" in texte:
            print(texte)
finally:
    classeur.close()

# Un formulaire d'un autre mois resté dans le dossier n'a pas été importé.
autres = sorted(
    p.name for p in chemin_form.parent.glob("*.xlsx")
    if p != chemin_form and not p.name.startswith("~$")
)
print(len(autres), "autres formulaires en attente", ", ".join(autres))

In [ ]:
# IMPORT FORMULAIRE

from frigeo.controles import lire_fichier_controles
from frigeo.formulaire_controles import (
    chemin_formulaire,
    empreinte_controles,
    importer_formulaire,
)

# Enregistrer et fermer le formulaire dans Excel avant d'exécuter cette cellule.
# Règles en vigueur avant l'import, que la cellule suivante compare au formulaire.
contenu_controles = lire_fichier_controles()
empreinte = empreinte_controles()
chemin_form = chemin_formulaire(PERIODE_FIN)
formulaire = None
if not chemin_form.exists():
    print("Aucun formulaire à importer,", chemin_form)
else:
    # Un formulaire mal rempli arrête ici, avec la liste des lignes à corriger.
    formulaire = importer_formulaire(
        chemin_form, dictionnaire, contenu_controles, empreinte=empreinte
    )
    print(len(formulaire.changements), "règles changées par le formulaire,",
          len(formulaire.avertissements), "avertissements")
    for avertissement in formulaire.avertissements:
        print(" ", avertissement)
    display(formulaire.changements)

In [ ]:
# ECRITURE FORMULAIRE

from frigeo.controles import chemin_regles_complementaires
from frigeo.formulaire_controles import appliquer_formulaire

if formulaire is None:
    print("Aucun formulaire importé, config/controles.yaml non réécrit.")
else:
    # Le formulaire est archivé dans data/formulaires/appliques, puis le fichier est
    # réécrit après une copie datée dans data/config/sauvegardes.
    application = appliquer_formulaire(
        formulaire, chemin_form, contenu_controles, empreinte=empreinte
    )
    if application.archive is None:
        print("Aucune règle changée, config/controles.yaml non réécrit, formulaire "
              "laissé en place.")
    else:
        print("Réécrit, à commiter,", chemin_regles_complementaires())
        print("Copie de l'ancien fichier", application.sauvegarde)
        print("Formulaire archivé dans", application.archive)
    # Une seconde exécution de la cellule ne doit pas appliquer le formulaire deux fois.
    formulaire = None

In [ ]:
%%time
# CONTROLES

from frigeo.controles import (
    charger_regles_complementaires,
    chemin_regles_complementaires,
    controler,
    regles_sans_controle,
)

# Règles données par le métier dans config/controles.yaml (aucune si le fichier est absent).
complementaires = charger_regles_complementaires()
nb_complementaires = sum(
    len(types) for colonnes in complementaires.values() for types in colonnes.values()
)
print(nb_complementaires, "règles complémentaires lues dans", chemin_regles_complementaires())

ecarts, bilan = controler(tables, dictionnaire, complementaires=complementaires)
sans_controle = regles_sans_controle(dictionnaire)
print(len(bilan), "contrôles,", len(sans_controle), "règles sans contrôle")
print(len(ecarts), "écarts, dont", int(bilan["anomalies"].sum()), "anomalies et",
      int(bilan["a_arbitrer"].sum()), "à arbitrer")
print(int(bilan["non_revues"].sum()), "cellules admises mais non revues")
display(
    bilan.groupby(["regle", "statut_regle"], dropna=False)[
        ["controlees", "anomalies", "a_arbitrer", "non_revues"]
    ].sum()
)

In [ ]:
# DETAIL PAR CONTROLE

a_voir = bilan[(bilan["anomalies"] > 0) | (bilan["a_arbitrer"] > 0)]
print(len(a_voir), "contrôles avec des anomalies ou des écarts à arbitrer")
display(a_voir.drop(columns=["non_revues"]))

# Une règle « à décider » ne contrôle rien tant que le métier n'a pas tranché.
a_decider = sans_controle[sans_controle["regle_en_vigueur"] == "à décider"]
print(len(a_decider), "règles « à décider », colonnes non contrôlées sur ce point")
display(a_decider)

In [ ]:
# COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite
from frigeo.controles import VALEUR_MASQUEE

sensibles = charger_sensibilite()
sur_sensible = pd.Series(
    [colonne in sensibles.get(table, ()) for table, colonne in zip(ecarts["table"], ecarts["colonne"])],
    index=ecarts.index,
    dtype=bool,
)
print(sum(len(colonnes) for colonnes in sensibles.values()), "colonnes sensibles,",
      int(sur_sensible.sum()), "écarts sur ces colonnes")
print("valeurs non masquées sur ces colonnes", int((ecarts.loc[sur_sensible, "valeur"] != VALEUR_MASQUEE).sum()))
print("valeurs masquées sur les autres colonnes", int((ecarts.loc[~sur_sensible, "valeur"] == VALEUR_MASQUEE).sum()))
display(
    ecarts[sur_sensible].groupby(["table", "colonne", "regle", "constat"]).size().rename("ecarts").to_frame()
)

In [ ]:
%%time
# DOUBLONS

from frigeo.controles import ISSUE_A_ARBITRER, ISSUE_ANOMALIE, chemin_regles_complementaires
from frigeo.doublons import charger_declarations_doublons, detecter_doublons, doublons_non_cherches

# Déclarations du métier dans config/controles.yaml, sous la clé doublons (aucune si absente) :
# sans clé déclarée, seuls les doublons exacts d'une table sont cherchés.
declarations = charger_declarations_doublons()
print(len(declarations), "tables avec des doublons déclarés dans", chemin_regles_complementaires())

doublons, bilan_doublons = detecter_doublons(tables, declarations)
non_cherches = doublons_non_cherches(tables, declarations)
print(len(bilan_doublons), "contrôles de doublons,", len(non_cherches), "constats non cherchés")
print(len(doublons), "lignes en doublon, dont",
      int((doublons["issue"] == ISSUE_ANOMALIE).sum()), "en anomalie et",
      int((doublons["issue"] == ISSUE_A_ARBITRER).sum()), "à arbitrer")
display(bilan_doublons[bilan_doublons["groupes"] > 0])
display(non_cherches)

In [ ]:
# DOUBLONS SUR COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite
from frigeo.controles import VALEUR_MASQUEE
from frigeo.doublons import CONSTAT_VALEUR

sensibles = charger_sensibilite()


def _touche_sensible(table, colonnes):
    return any(colonne in sensibles.get(table, ()) for colonne in colonnes)


# La valeur affichée vient des colonnes comparées, la clé des colonnes de la clé déclarée.
valeur_sensible = pd.Series(
    [
        constat == CONSTAT_VALEUR and _touche_sensible(table, colonnes.split(", "))
        for table, constat, colonnes in zip(doublons["table"], doublons["constat"], doublons["colonnes"])
    ],
    index=doublons.index,
    dtype=bool,
)
cle_sensible = pd.Series(
    [_touche_sensible(table, declarations.get(table, {}).get("cle", [])) for table in doublons["table"]],
    index=doublons.index,
    dtype=bool,
)
print("valeurs en double sur des colonnes sensibles", int(valeur_sensible.sum()),
      ", dont non masquées", int((doublons.loc[valeur_sensible, "valeur"] != VALEUR_MASQUEE).sum()))
print("clés sensibles", int(cle_sensible.sum()),
      ", dont non masquées", int((doublons.loc[cle_sensible, "cle"] != VALEUR_MASQUEE).sum()))
print("valeurs ou clés masquées ailleurs",
      int(((doublons["valeur"] == VALEUR_MASQUEE) & ~valeur_sensible).sum()
          + ((doublons["cle"] == VALEUR_MASQUEE) & ~cle_sensible).sum()))
display(doublons[valeur_sensible].groupby(["table", "colonnes", "issue"]).size().rename("lignes").to_frame())